# SSTW videoseal 官方 baseline 服务器 workflow Colab 入口

该 Notebook 只承担 Drive 挂载、认证、环境安装和参数选择。正式阶段顺序、算法、检测、统计、门禁与产物生成全部由 `scripts/run_generative_video_server_workflow.py` 执行。相同命令可脱离 Notebook 在普通 GPU 服务器运行。


In [ ]:
SSTW_WORKFLOW_PROFILE_VALUE = 'probe_paper'  # 可改为 'pilot_paper' 或 'full_paper'
# 1. 挂载 Google Drive; GPU、依赖和代码状态由服务器 CLI 的 fail-closed preflight 复核。
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
import subprocess
import sys

DRIVE_PROJECT_ROOT = os.environ.get('SSTW_DRIVE_PROJECT_ROOT', '/content/drive/MyDrive/SSTW')
REPO_URL = os.environ.get('SSTW_REPO_URL', 'https://github.com/RICHAAARC/SSTW.git')
REPO_DIR = os.environ.get('SSTW_REPO_DIR', '/content/SSTW')
REPO_REF = os.environ.get('SSTW_REPO_REF', '').strip()
WORKFLOW_PROFILE = globals().get('SSTW_WORKFLOW_PROFILE_VALUE', 'probe_paper').strip()
os.environ['SSTW_WORKFLOW_PROFILE'] = WORKFLOW_PROFILE
print(subprocess.getoutput('nvidia-smi'))


In [ ]:
# 2. 获取仓库代码并冻结本次实际执行 commit。
repo_path = Path(REPO_DIR)
if not repo_path.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR], check=True)
if REPO_REF:
    subprocess.run(['git', 'fetch', '--tags', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
RESOLVED_REPO_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
print('SSTW repository commit =', RESOLVED_REPO_COMMIT)


In [ ]:
# 3. 安装受版本锁约束的公共论文运行环境。
# 不使用 -U 或 Git 分支依赖, 防止 Colab 与普通 GPU 服务器产生依赖漂移。
%pip install --requirement requirements/paper_runtime_lock.txt


In [ ]:
# 4. 可选 Hugging Face 认证; token 不进入实验 records 或运行清单。
from huggingface_hub import login

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not provided; gated model 将由服务器 preflight 阻断。')


In [ ]:
# 5. 调用与普通 GPU 服务器完全相同的唯一 workflow CLI。
from workflows.streaming_command import run_streaming_command

NOTEBOOK_ROLE = 'external_baseline_formal_scoring'
SERVER_PIPELINE = 'external_baseline_references'
server_command = [
    sys.executable,
    'scripts/run_generative_video_server_workflow.py',
    '--project-root', DRIVE_PROJECT_ROOT,
    '--repo-root', REPO_DIR,
    '--workflow-profile', WORKFLOW_PROFILE,
    '--pipeline', SERVER_PIPELINE,
    '--local-workspace-root', '/content/SSTW_stage_workspace',
    '--local-package-cache-root', '/content/SSTW_stage_packages',
    '--decision-output', '/content/sstw_server_workflow_decision.json',
]
if os.environ.get('SSTW_MODEL_REVISION'):
    server_command.extend(['--model-revision', os.environ['SSTW_MODEL_REVISION']])
if os.environ.get('SSTW_CROSS_MODEL_REVISION'):
    server_command.extend(['--cross-model-revision', os.environ['SSTW_CROSS_MODEL_REVISION']])
if os.environ.get('SSTW_INCLUDE_VIDEOS_IN_PACKAGE', 'true').lower() != 'true':
    server_command.append('--exclude-videos')
server_command.extend(['--baseline-id', 'videoseal'])

print('server workflow pipeline =', SERVER_PIPELINE)
result = run_streaming_command(server_command)
if result.returncode != 0:
    raise RuntimeError(f'服务器 workflow 失败, return_code={result.returncode}')
